# Falsabilidad como estructura (P1-b.3b)
El agente debe fijar **qué resultado refutaría** su hipótesis al proponerla, y para pedir confirmarla debe aportar al menos un intento
de refutación: una consulta posterior a la hipótesis. Tú ves ambas cosas al aprobar. Ver `docs/falsabilidad.md`.

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from dfir_copilot.agent.graph import build_agent
from dfir_copilot.agent.llm import make_llm
from dfir_copilot.cases import CaseWorkspace

ws = CaseWorkspace.open("IDOR-INVOICES-2020Q4-TZ-SANTIAGO")
real = ws.engine(timeout_s=120)
engine, ps = ws.pseudonymized(timeout_s=120)
ledger = ws.ledger(real)
agent = build_agent(engine, ledger, make_llm(), max_steps=14)

## 1. Preguntar (gasta API)
Pide explícitamente el criterio de refutación y el intento; el prompt de sistema ya se lo exige al modelo.

In [ ]:
r = agent.ask("¿Hay un grupo coordinado que enumera facturas ajenas en /invoices/search? Formula la hipótesis con su criterio "
              "de refutación, pruébala e intenta refutarla antes de pedir su confirmación.")
print(r.status, "| pasos:", r.steps, "| tokens:", r.tokens)
print(ps.reveal_any(r.answer) if r.answer else "(pausado: hay una hipótesis pendiente de tu aprobación)")

## 2. Lo que ves antes de decidir
Todo en alias en el ledger; aquí se traduce **solo para ti** con `reveal_any`.

In [ ]:
for req in agent.pending():
    print(f"{req['hypothesis_id']}: {req['estado_actual']} -> {req['estado_solicitado']}")
    print("  Hipótesis:            ", ps.reveal_any(req["hipotesis"]))
    print("  Criterio de refutación:", ps.reveal_any(req["criterio_de_refutacion"]))
    print("  Justificación:        ", ps.reveal_any(req["justificacion"]))
    for i, a in enumerate(req["intentos_de_refutacion"], 1):
        print(f"  Intento {i}: [{a['ref']}] {ps.reveal_any(a['detalle'])[:300]}")
        print("      la habría refutado si:", ps.reveal_any(a["habria_refutado_si"]))
        print("      observado:            ", ps.reveal_any(a["observado"]))

## 3. Decidir
Pregúntate: ¿esa consulta **podía** haber salido distinto a como salió? Si el intento solo repite lo que ya apoyaba la hipótesis, rechaza
y dilo en la nota. **Escribe la nota con alias**; si pones valores reales se traducen solos, pero lo ambiguo se bloquea.

In [ ]:
# r = agent.resolve("approve", "Revisé el intento de refutación: la consulta podía fallar y no falló.")
# r = agent.resolve("reject", "El intento no puede refutar la hipótesis: repite lo que ya la apoyaba.")
# print(r.status, "|", r.sent)

## 4. Cómo se mide
Todo sale del ledger. La proporción de rechazos tras presentar intentos mide la calidad con la que el agente prueba sus hipótesis.

In [ ]:
rows = [{"id": h["hypothesis_id"], "estado": h["status"], "con_criterio": bool(h.get("falsifier")),
         "intentos_registrados": len(h.get("refutation_checks", []))} for h in agent.book.all()]
display(pd.DataFrame(rows))
decisions = [e["data"]["decision"] for e in ledger.entries("hypothesis_update") if e["data"]["decision"] in ("approved", "rejected")]
print("Decisiones del analista:", {d: decisions.count(d) for d in sorted(set(decisions))})

In [ ]:
r = agent.resolve("approve",
    "Aprobada solo en lo observable en los logs: grupo cerrado de cuatro cuentas con cinco IPs exclusivas, "
    "IDs que nadie más consulta y orden no secuencial. NO establece que las facturas sean ajenas (falta el mapa "
    "factura-propietario) ni que se devolviera contenido (un 200 no lo prueba).")
print(r.status, "|", r.sent)